In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import lightgbm as lgb
import xgboost as xgb
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# 딥러닝 라이브러리
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    from torch.nn.utils.rnn import pad_sequence
    GPU_AVAILABLE = torch.cuda.is_available()
    device = torch.device('cuda' if GPU_AVAILABLE else 'cpu')
except ImportError:
    GPU_AVAILABLE = False
    device = 'cpu'

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False

print(f"GPU 사용 가능: {GPU_AVAILABLE}")
print(f"Prophet 사용 가능: {PROPHET_AVAILABLE}")
print(f"사용 디바이스: {device}")

# 전역 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
np.random.seed(42)
torch.manual_seed(42) if GPU_AVAILABLE else None

# 전역 변수로 결과 저장
GLOBAL_RESULTS = {
    'original_df': None,
    'normalized_df': None,
    'predictions': None,
    'ensemble_pred': None,
    'covid_data': None,
    'performance_metrics': None
}

class DataPreprocessor:
    def __init__(self):
        self.label_encoders = {}
        self.scalers = {}
        
    def load_and_prepare_data(self):
        """데이터 로드 및 기본 전처리"""
        df = pd.read_csv('외국인입국자_전처리완료_딥러닝용.csv', encoding='utf-8')
        
        # 레이블 인코딩
        categorical_cols = ['국적', '목적', '계절']
        for col in categorical_cols:
            le = LabelEncoder()
            df[f'{col}_encoded'] = le.fit_transform(df[col])
            self.label_encoders[col] = le
        
        # 날짜 정보 생성
        df['date'] = pd.to_datetime(df[['연도', '월']].assign(day=1))
        df = df.sort_values('date').reset_index(drop=True)
        
        # 데이터 분할
        train_data = df[df['연도'] <= 2019].copy()
        covid_data = df[(df['연도'] >= 2020) & (df['연도'] <= 2023)].copy()
        
        print(f"전체 데이터: {len(df):,}개")
        print(f"학습 데이터: {len(train_data):,}개 (2005-2019)")
        print(f"예측 대상: {len(covid_data):,}개 (2020-2023)")
        
        return df, train_data, covid_data
    
    def prepare_features(self, data, target_col='입국자수'):
        """피처 준비 및 스케일링"""
        feature_cols = [
            '국적_encoded', '목적_encoded', '연도', '월', '분기', '계절_encoded',
            '시계열순서', '입국자수_1개월전', '입국자수_3개월전', '입국자수_12개월전',
            '입국자수_3개월평균', '입국자수_12개월평균', '전년동월대비증감률'
        ]
        
        # 결측치 처리
        X = data[feature_cols].copy()
        X = X.fillna(method='ffill').fillna(method='bfill').fillna(0)
        
        # 이상치 처리 (IQR 방법)
        if target_col in data.columns:
            y = data[target_col].copy()
            Q1 = y.quantile(0.25)
            Q3 = y.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            y = np.clip(y, lower_bound, upper_bound)
        else:
            y = None
        
        return X, y, feature_cols
    
    def create_time_series_data(self, data, seq_length=24):
        """시계열 딥러닝용 데이터 준비"""
        # 월별 집계
        monthly_data = data.groupby(['연도', '월']).agg({
            '입국자수': 'sum',
            '국적_encoded': 'mean',
            '목적_encoded': 'mean',
            '계절_encoded': 'mean'
        }).reset_index()
        
        # 시퀀스 생성
        sequences = []
        targets = []
        features = []
        
        for i in range(len(monthly_data) - seq_length):
            seq = monthly_data['입국자수'].iloc[i:i+seq_length].values
            target = monthly_data['입국자수'].iloc[i+seq_length]
            feat = monthly_data[['국적_encoded', '목적_encoded', '계절_encoded']].iloc[i+seq_length].values
            
            sequences.append(seq)
            targets.append(target)
            features.append(feat)
        
        return np.array(sequences), np.array(targets), np.array(features)

class TimeSeriesModels:
    def __init__(self):
        self.models = {}
        
    def fit_prophet(self, train_data):
        """Prophet 모델"""
        if not PROPHET_AVAILABLE:
            print("Prophet 라이브러리가 설치되지 않았습니다.")
            return None
        
        # 월별 데이터 준비
        monthly_data = train_data.groupby(['연도', '월'])['입국자수'].sum().reset_index()
        monthly_data['ds'] = pd.to_datetime(monthly_data[['연도', '월']].assign(day=1))
        monthly_data['y'] = monthly_data['입국자수']
        
        # Prophet 모델 설정
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.1,
            seasonality_prior_scale=10
        )
        
        model.fit(monthly_data[['ds', 'y']])
        return model
    
    def fit_sarimax(self, train_data):
        """SARIMAX 모델"""
        monthly_data = train_data.groupby(['연도', '월'])['입국자수'].sum()
        
        # 정상성 검정
        adf_result = adfuller(monthly_data)
        print(f"SARIMAX - ADF 검정 p-value: {adf_result[1]:.4f}")
        
        # 최적 파라미터 찾기 (간소화)
        best_aic = float('inf')
        best_params = None
        
        param_combinations = [(1,1,1), (1,1,2), (2,1,1)]
        seasonal_combinations = [(1,1,1,12), (1,1,2,12)]
        
        for p,d,q in param_combinations:
            for P,D,Q,s in seasonal_combinations:
                try:
                    model = SARIMAX(monthly_data, order=(p,d,q), seasonal_order=(P,D,Q,s))
                    fitted_model = model.fit(disp=False)
                    if fitted_model.aic < best_aic:
                        best_aic = fitted_model.aic
                        best_params = ((p,d,q), (P,D,Q,s))
                except:
                    continue
        
        if best_params:
            print(f"SARIMAX 최적 파라미터: {best_params}, AIC: {best_aic:.2f}")
            final_model = SARIMAX(monthly_data, order=best_params[0], seasonal_order=best_params[1])
            return final_model.fit(disp=False)
        else:
            model = SARIMAX(monthly_data, order=(1,1,1), seasonal_order=(1,1,1,12))
            return model.fit(disp=False)
    
    def fit_exponential_smoothing(self, train_data):
        """지수 평활법"""
        monthly_data = train_data.groupby(['연도', '월'])['입국자수'].sum()
        
        # 기본 설정으로 빠르게 학습
        try:
            model = ExponentialSmoothing(monthly_data, trend='add', seasonal='add', seasonal_periods=12)
            fitted_model = model.fit()
            print(f"지수평활법 AIC: {fitted_model.aic:.2f}")
            return fitted_model
        except:
            model = ExponentialSmoothing(monthly_data, trend='add', seasonal=None)
            return model.fit()

class MachineLearningModels:
    def __init__(self):
        self.models = {}
        
    def fit_lightgbm(self, X_train, y_train):
        """LightGBM - 간소화된 튜닝"""
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [6, 8],
            'learning_rate': [0.05, 0.1]
        }
        
        lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
        tscv = TimeSeriesSplit(n_splits=3)
        
        grid_search = GridSearchCV(lgb_model, param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        
        print(f"LightGBM 최적 파라미터: {grid_search.best_params_}")
        return grid_search.best_estimator_
    
    def fit_xgboost(self, X_train, y_train):
        """XGBoost - 간소화된 튜닝"""
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [4, 6],
            'learning_rate': [0.05, 0.1]
        }
        
        xgb_model = xgb.XGBRegressor(random_state=42, verbosity=0)
        tscv = TimeSeriesSplit(n_splits=3)
        
        grid_search = GridSearchCV(xgb_model, param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        
        print(f"XGBoost 최적 파라미터: {grid_search.best_params_}")
        return grid_search.best_estimator_
    
    def fit_random_forest(self, X_train, y_train):
        """Random Forest - 간소화된 튜닝"""
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [10, 15],
            'min_samples_split': [2, 5]
        }
        
        rf_model = RandomForestRegressor(random_state=42)
        tscv = TimeSeriesSplit(n_splits=3)
        
        grid_search = GridSearchCV(rf_model, param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        
        print(f"RandomForest 최적 파라미터: {grid_search.best_params_}")
        return grid_search.best_estimator_

# 딥러닝 모델들 (간소화된 버전)
class SimpleDeepLearningModels:
    def __init__(self, device='cpu'):
        self.device = device
    
    def create_sequences(self, data, seq_length=12):
        """시퀀스 데이터 생성 (단순화)"""
        monthly_data = data.groupby(['연도', '월'])['입국자수'].sum().values
        
        sequences = []
        targets = []
        
        for i in range(len(monthly_data) - seq_length):
            sequences.append(monthly_data[i:i+seq_length])
            targets.append(monthly_data[i+seq_length])
        
        return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super(SimpleLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out

def train_simple_model(model, X_train, y_train, epochs=30):
    """간단한 딥러닝 모델 훈련"""
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # 텐서 변환
    X_tensor = torch.FloatTensor(X_train).unsqueeze(-1).to(device)
    y_tensor = torch.FloatTensor(y_train).to(device)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs.squeeze(), y_tensor)
        loss.backward()
        optimizer.step()
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    
    return model

def run_covid_normalization():
    """코로나 정상화 모델 실행 (파일 생성 제외)"""
    
    print("=== 코로나 정상화 모델링 시작 ===")
    
    # 1. 데이터 전처리
    preprocessor = DataPreprocessor()
    df, train_data, covid_data = preprocessor.load_and_prepare_data()
    
    X_train, y_train, feature_cols = preprocessor.prepare_features(train_data)
    X_covid, _, _ = preprocessor.prepare_features(covid_data)
    
    # 스케일링
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_covid_scaled = scaler.transform(X_covid)
    
    print(f"피처 수: {len(feature_cols)}")
    print(f"학습 데이터 크기: {X_train_scaled.shape}")
    
    # 결과 저장
    predictions = {}
    
    # 2. 시계열 모델들
    print("\n=== 시계열 모델 학습 ===")
    ts_models = TimeSeriesModels()
    
    # Prophet
    if PROPHET_AVAILABLE:
        print("Prophet 학습 중...")
        try:
            prophet_model = ts_models.fit_prophet(train_data)
            if prophet_model:
                future_dates = pd.date_range(start='2020-01-01', end='2023-12-01', freq='MS')
                future_df = pd.DataFrame({'ds': future_dates})
                prophet_pred = prophet_model.predict(future_df)
                
                # 월별 예측값을 일별로 분배
                monthly_preds = prophet_pred['yhat'].values
                daily_preds = []
                for i, pred in enumerate(monthly_preds):
                    year = 2020 + i // 12
                    month = (i % 12) + 1
                    month_data = covid_data[(covid_data['연도'] == year) & (covid_data['월'] == month)]
                    if len(month_data) > 0:
                        daily_pred = max(pred / len(month_data), 0)  # 음수 방지
                        daily_preds.extend([daily_pred] * len(month_data))
                
                # 길이 맞추기
                if len(daily_preds) > len(covid_data):
                    daily_preds = daily_preds[:len(covid_data)]
                elif len(daily_preds) < len(covid_data):
                    daily_preds.extend([daily_preds[-1]] * (len(covid_data) - len(daily_preds)))
                
                predictions['Prophet'] = np.array(daily_preds)
                print("Prophet 완료")
        except Exception as e:
            print(f"Prophet 에러: {e}")
    
    # SARIMAX
    print("SARIMAX 학습 중...")
    try:
        sarimax_model = ts_models.fit_sarimax(train_data)
        if sarimax_model:
            sarimax_pred = sarimax_model.forecast(steps=48)
            
            # 일별로 분배
            daily_preds = []
            for i, pred in enumerate(sarimax_pred):
                year = 2020 + i // 12
                month = (i % 12) + 1
                month_data = covid_data[(covid_data['연도'] == year) & (covid_data['월'] == month)]
                if len(month_data) > 0:
                    daily_pred = max(pred / len(month_data), 0)
                    daily_preds.extend([daily_pred] * len(month_data))
            
            if len(daily_preds) > len(covid_data):
                daily_preds = daily_preds[:len(covid_data)]
            elif len(daily_preds) < len(covid_data):
                daily_preds.extend([daily_preds[-1]] * (len(covid_data) - len(daily_preds)))
            
            predictions['SARIMAX'] = np.array(daily_preds)
            print("SARIMAX 완료")
    except Exception as e:
        print(f"SARIMAX 에러: {e}")
    
    # Exponential Smoothing
    print("지수평활법 학습 중...")
    try:
        es_model = ts_models.fit_exponential_smoothing(train_data)
        if es_model:
            es_pred = es_model.forecast(steps=48)
            
            # 일별로 분배
            daily_preds = []
            for i, pred in enumerate(es_pred):
                year = 2020 + i // 12
                month = (i % 12) + 1
                month_data = covid_data[(covid_data['연도'] == year) & (covid_data['월'] == month)]
                if len(month_data) > 0:
                    daily_pred = max(pred / len(month_data), 0)
                    daily_preds.extend([daily_pred] * len(month_data))
            
            if len(daily_preds) > len(covid_data):
                daily_preds = daily_preds[:len(covid_data)]
            elif len(daily_preds) < len(covid_data):
                daily_preds.extend([daily_preds[-1]] * (len(covid_data) - len(daily_preds)))
            
            predictions['ExponentialSmoothing'] = np.array(daily_preds)
            print("지수평활법 완료")
    except Exception as e:
        print(f"지수평활법 에러: {e}")
    
    # 3. 머신러닝 모델들
    print("\n=== 머신러닝 모델 학습 ===")
    ml_models = MachineLearningModels()
    
    # LightGBM
    print("LightGBM 학습 중...")
    lgb_model = ml_models.fit_lightgbm(X_train_scaled, y_train)
    lgb_pred = np.maximum(lgb_model.predict(X_covid_scaled), 0)  # 음수 방지
    predictions['LightGBM'] = lgb_pred
    
    # XGBoost
    print("XGBoost 학습 중...")
    xgb_model = ml_models.fit_xgboost(X_train_scaled, y_train)
    xgb_pred = np.maximum(xgb_model.predict(X_covid_scaled), 0)
    predictions['XGBoost'] = xgb_pred
    
    # Random Forest
    print("RandomForest 학습 중...")
    rf_model = ml_models.fit_random_forest(X_train_scaled, y_train)
    rf_pred = np.maximum(rf_model.predict(X_covid_scaled), 0)
    predictions['RandomForest'] = rf_pred
    
    # 4. 간단한 딥러닝 모델들
    print("\n=== 딥러닝 모델 학습 ===")
    
    dl_models = SimpleDeepLearningModels(device)
    X_seq, y_seq = dl_models.create_sequences(train_data, seq_length=12)
    
    if len(X_seq) > 5:  # 최소 데이터 확인
        split_idx = int(len(X_seq) * 0.8)
        X_train_seq, y_train_seq = X_seq[:split_idx], y_seq[:split_idx]
        
        # LSTM 1
        print("LSTM 학습 중...")
        try:
            lstm_model = SimpleLSTM()
            lstm_model = train_simple_model(lstm_model, X_train_seq, y_train_seq, epochs=30)
            
            # 예측
            lstm_model.eval()
            with torch.no_grad():
                last_sequence = X_seq[-1:].reshape(1, 12, 1)
                lstm_predictions = []
                
                for _ in range(48):
                    pred = lstm_model(torch.FloatTensor(last_sequence).to(device))
                    pred_value = max(pred.cpu().item(), 0)
                    lstm_predictions.append(pred_value)
                    
                    # 시퀀스 업데이트
                    last_sequence = np.roll(last_sequence, -1, axis=1)
                    last_sequence[0, -1, 0] = pred_value
            
            # 일별로 분배
            daily_preds = []
            for i, pred in enumerate(lstm_predictions):
                year = 2020 + i // 12
                month = (i % 12) + 1
                month_data = covid_data[(covid_data['연도'] == year) & (covid_data['월'] == month)]
                if len(month_data) > 0:
                    daily_pred = pred / len(month_data)
                    daily_preds.extend([daily_pred] * len(month_data))
            
            if len(daily_preds) > len(covid_data):
                daily_preds = daily_preds[:len(covid_data)]
            elif len(daily_preds) < len(covid_data):
                daily_preds.extend([daily_preds[-1]] * (len(covid_data) - len(daily_preds)))
            
            predictions['LSTM'] = np.array(daily_preds)
            print("LSTM 완료")
        except Exception as e:
            print(f"LSTM 에러: {e}")
        
        # 추가 딥러닝 모델들 (LSTM 변형)
        for i in range(2, 7):  # LSTM2~LSTM6
            try:
                model_name = f"LSTM_{i}"
                print(f"{model_name} 학습 중...")
                
                # 하이퍼파라미터 조정
                hidden_size = 30 + i * 10
                lstm_variant = SimpleLSTM(hidden_size=hidden_size)
                lstm_variant = train_simple_model(lstm_variant, X_train_seq, y_train_seq, epochs=20)
                
                # 예측
                lstm_variant.eval()
                with torch.no_grad():
                    last_sequence = X_seq[-1:].reshape(1, 12, 1)
                    variant_predictions = []
                    
                    for _ in range(48):
                        pred = lstm_variant(torch.FloatTensor(last_sequence).to(device))
                        pred_value = max(pred.cpu().item(), 0)
                        variant_predictions.append(pred_value)
                        
                        last_sequence = np.roll(last_sequence, -1, axis=1)
                        last_sequence[0, -1, 0] = pred_value
                
                # 일별로 분배
                daily_preds = []
                for j, pred in enumerate(variant_predictions):
                    year = 2020 + j // 12
                    month = (j % 12) + 1
                    month_data = covid_data[(covid_data['연도'] == year) & (covid_data['월'] == month)]
                    if len(month_data) > 0:
                        daily_pred = pred / len(month_data)
                        daily_preds.extend([daily_pred] * len(month_data))
                
                if len(daily_preds) > len(covid_data):
                    daily_preds = daily_preds[:len(covid_data)]
                elif len(daily_preds) < len(covid_data):
                    daily_preds.extend([daily_preds[-1]] * (len(covid_data) - len(daily_preds)))
                
                predictions[model_name] = np.array(daily_preds)
                print(f"{model_name} 완료")
            except Exception as e:
                print(f"{model_name} 에러: {e}")
    
    print(f"\n=== 총 {len(predictions)}개 모델 학습 완료 ===")
    
    # 유효한 예측값만 필터링
    target_length = len(covid_data)
    valid_predictions = {}
    
    for name, pred in predictions.items():
        if len(pred) == target_length:
            pred_clipped = np.maximum(pred, 0)  # 음수값 제거
            valid_predictions[name] = pred_clipped
            print(f"{name}: 평균 {np.mean(pred_clipped):,.0f}명")
        else:
            print(f"{name}: 길이 불일치 - 제외")
    
    if len(valid_predictions) == 0:
        print("유효한 예측값이 없습니다.")
        return None
    
    # 앙상블 예측
    ensemble_pred = np.mean(list(valid_predictions.values()), axis=0)
    
    # 정상화된 데이터프레임 생성 (저장하지 않음)
    df_normalized = df.copy()
    covid_indices = df_normalized[(df_normalized['연도'] >= 2020) & (df_normalized['연도'] <= 2023)].index
    
    if len(covid_indices) == len(ensemble_pred):
        df_normalized.loc[covid_indices, '입국자수'] = ensemble_pred.astype(int)
    else:
        min_length = min(len(covid_indices), len(ensemble_pred))
        df_normalized.loc[covid_indices[:min_length], '입국자수'] = ensemble_pred[:min_length].astype(int)
    
    # 전역 변수에 결과 저장
    GLOBAL_RESULTS['original_df'] = df
    GLOBAL_RESULTS['normalized_df'] = df_normalized
    GLOBAL_RESULTS['predictions'] = valid_predictions
    GLOBAL_RESULTS['ensemble_pred'] = ensemble_pred
    GLOBAL_RESULTS['covid_data'] = covid_data
    
    return df, df_normalized, valid_predictions, ensemble_pred, covid_data

def show_analysis_graphs():
    """분석 그래프 표시"""
    
    if GLOBAL_RESULTS['original_df'] is None:
        print("먼저 run_covid_normalization()을 실행해주세요.")
        return
    
    df = GLOBAL_RESULTS['original_df']
    df_normalized = GLOBAL_RESULTS['normalized_df']
    valid_predictions = GLOBAL_RESULTS['predictions']
    ensemble_pred = GLOBAL_RESULTS['ensemble_pred']
    covid_data = GLOBAL_RESULTS['covid_data']
    
    # 시각화
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # 1. 전체 트렌드
    monthly_original = df.groupby(['연도', '월'])['입국자수'].sum()
    monthly_normalized = df_normalized.groupby(['연도', '월'])['입국자수'].sum()
    
    axes[0,0].plot(range(len(monthly_original)), monthly_original, label='원본 데이터', alpha=0.8, linewidth=2)
    axes[0,0].plot(range(len(monthly_normalized)), monthly_normalized, label='정상화 데이터', alpha=0.8, linewidth=2)
    axes[0,0].axvspan(180, 228, alpha=0.2, color='red', label='코로나 기간')
    axes[0,0].set_title('전체 입국자 수 변화', fontsize=14, fontweight='bold')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. 코로나 기간 확대
    covid_start_idx = 180
    covid_end_idx = min(228, len(monthly_original))
    covid_original = monthly_original[covid_start_idx:covid_end_idx]
    covid_normalized = monthly_normalized[covid_start_idx:covid_end_idx]
    
    axes[0,1].plot(range(len(covid_original)), covid_original, label='원본', marker='o', linewidth=2)
    axes[0,1].plot(range(len(covid_normalized)), covid_normalized, label='정상화', marker='s', linewidth=2)
    axes[0,1].set_title('코로나 기간 (2020-2023) 상세 비교', fontsize=14, fontweight='bold')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. 모델별 예측 평균값
    model_names = list(valid_predictions.keys())
    pred_means = [np.mean(pred) for pred in valid_predictions.values()]
    
    bars = axes[0,2].bar(range(len(model_names)), pred_means, color='skyblue', alpha=0.7)
    axes[0,2].set_xticks(range(len(model_names)))
    axes[0,2].set_xticklabels(model_names, rotation=45, ha='right')
    axes[0,2].set_title('모델별 평균 예측값', fontsize=14, fontweight='bold')
    axes[0,2].grid(True, alpha=0.3)
    
    # 값 표시
    for bar, mean_val in zip(bars, pred_means):
        axes[0,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(pred_means)*0.01,
                      f'{mean_val:,.0f}', ha='center', va='bottom', fontsize=10)
    
    # 4. 연도별 비교
    yearly_original = df.groupby('연도')['입국자수'].sum()
    yearly_normalized = df_normalized.groupby('연도')['입국자수'].sum()
    
    years = yearly_original.index
    axes[1,0].plot(years, yearly_original, label='원본', marker='o', linewidth=3, markersize=8)
    axes[1,0].plot(years, yearly_normalized, label='정상화', marker='s', linewidth=3, markersize=8)
    axes[1,0].axvspan(2020, 2023, alpha=0.2, color='red', label='코로나 기간')
    axes[1,0].set_title('연도별 입국자 수', fontsize=14, fontweight='bold')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    axes[1,0].set_xlabel('연도')
    axes[1,0].set_ylabel('입국자 수')
    
    # 5. 모델별 예측 분포
    all_predictions = list(valid_predictions.values())
    if len(model_names) <= 10:  # 모델이 너무 많으면 생략
        axes[1,1].boxplot(all_predictions, labels=model_names)
        axes[1,1].set_title('모델별 예측값 분포', fontsize=14, fontweight='bold')
        axes[1,1].tick_params(axis='x', rotation=45)
        axes[1,1].grid(True, alpha=0.3)
    else:
        # 상위 5개 모델만 표시
        top_5_models = sorted(valid_predictions.items(), key=lambda x: np.mean(x[1]), reverse=True)[:5]
        top_5_names = [x[0] for x in top_5_models]
        top_5_preds = [x[1] for x in top_5_models]
        
        axes[1,1].boxplot(top_5_preds, labels=top_5_names)
        axes[1,1].set_title('상위 5개 모델 예측값 분포', fontsize=14, fontweight='bold')
        axes[1,1].tick_params(axis='x', rotation=45)
        axes[1,1].grid(True, alpha=0.3)
    
    # 6. 정상화 효과 비교
    pre_covid = yearly_original[yearly_original.index <= 2019].mean()
    covid_original_mean = yearly_original[(yearly_original.index >= 2020) & (yearly_original.index <= 2023)].mean()
    covid_normalized_mean = yearly_normalized[(yearly_normalized.index >= 2020) & (yearly_normalized.index <= 2023)].mean()
    
    categories = ['코로나 이전\n(2005-2019)', '코로나 기간\n(원본)', '코로나 기간\n(정상화)']
    values = [pre_covid, covid_original_mean, covid_normalized_mean]
    colors = ['green', 'red', 'blue']
    
    bars = axes[1,2].bar(categories, values, color=colors, alpha=0.7)
    axes[1,2].set_title('정상화 효과 비교', fontsize=14, fontweight='bold')
    axes[1,2].grid(True, alpha=0.3)
    
    # 값 표시
    for bar, val in zip(bars, values):
        axes[1,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                      f'{val:,.0f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def print_detailed_analysis():
    """상세 분석 결과 출력"""
    
    if GLOBAL_RESULTS['original_df'] is None:
        print("먼저 run_covid_normalization()을 실행해주세요.")
        return
    
    df = GLOBAL_RESULTS['original_df']
    df_normalized = GLOBAL_RESULTS['normalized_df']
    valid_predictions = GLOBAL_RESULTS['predictions']
    ensemble_pred = GLOBAL_RESULTS['ensemble_pred']
    covid_data = GLOBAL_RESULTS['covid_data']
    
    print("\n" + "="*80)
    print("상세 분석 결과")
    print("="*80)
    
    print(f"\n1. 학습된 모델 수: {len(valid_predictions)}개")
    for i, name in enumerate(valid_predictions.keys(), 1):
        print(f"   {i}. {name}")
    
    print(f"\n2. 앙상블 통계:")
    print(f"   - 평균 예측값: {np.mean(ensemble_pred):,.0f}명")
    print(f"   - 표준편차: {np.std(ensemble_pred):,.0f}명")
    print(f"   - 최소값: {np.min(ensemble_pred):,.0f}명")
    print(f"   - 최대값: {np.max(ensemble_pred):,.0f}명")
    
    print(f"\n3. 코로나 기간 총 정상화 입국자 수:")
    covid_total_original = df[(df['연도'] >= 2020) & (df['연도'] <= 2023)]['입국자수'].sum()
    covid_total_normalized = df_normalized[(df_normalized['연도'] >= 2020) & (df_normalized['연도'] <= 2023)]['입국자수'].sum()
    print(f"   - 원본: {covid_total_original:,.0f}명")
    print(f"   - 정상화: {covid_total_normalized:,.0f}명")
    print(f"   - 증가량: {covid_total_normalized - covid_total_original:,.0f}명 ({((covid_total_normalized/covid_total_original)-1)*100:.1f}% 증가)")
    
    print(f"\n4. 모델별 상세 예측 결과:")
    model_stats = []
    for name, pred in valid_predictions.items():
        stats = {
            'Model': name,
            'Mean': np.mean(pred),
            'Std': np.std(pred),
            'Median': np.median(pred),
            'CV': np.std(pred) / np.mean(pred)
        }
        model_stats.append(stats)
        print(f"   {name}:")
        print(f"     - 평균: {stats['Mean']:,.0f}명")
        print(f"     - 표준편차: {stats['Std']:,.0f}명")
        print(f"     - 중앙값: {stats['Median']:,.0f}명")
        print(f"     - 변동계수: {stats['CV']:.3f}")
    
    print(f"\n5. 정상화 품질 지표:")
    yearly_original = df.groupby('연도')['입국자수'].sum()
    yearly_normalized = df_normalized.groupby('연도')['입국자수'].sum()
    
    # 2019년 대비 회복률
    pre_covid_2019 = yearly_original[2019]
    recovery_2024 = yearly_normalized[yearly_normalized.index >= 2024].mean() if len(yearly_normalized[yearly_normalized.index >= 2024]) > 0 else yearly_normalized[2023]
    recovery_rate = (recovery_2024 / pre_covid_2019) * 100
    print(f"   - 2019년 대비 회복률: {recovery_rate:.1f}%")
    
    # 계절성 보존 확인
    monthly_original = df.groupby(['연도', '월'])['입국자수'].sum()
    monthly_normalized = df_normalized.groupby(['연도', '월'])['입국자수'].sum()
    
    covid_start_idx = 180
    covid_end_idx = min(228, len(monthly_original))
    
    original_seasonality = monthly_original[covid_start_idx:covid_end_idx].std()
    normalized_seasonality = monthly_normalized[covid_start_idx:covid_end_idx].std()
    seasonality_preservation = (normalized_seasonality / original_seasonality) * 100
    print(f"   - 계절성 보존도: {seasonality_preservation:.1f}%")
    
    # 6. 모델 순위 (변동계수 기준)
    print(f"\n6. 모델 안정성 순위 (변동계수 낮은 순):")
    sorted_models = sorted(model_stats, key=lambda x: x['CV'])
    for i, model in enumerate(sorted_models, 1):
        print(f"   {i}. {model['Model']}: CV = {model['CV']:.3f}")
    
    print("\n" + "="*80)
    
    return model_stats

def save_normalized_data(filename=None):
    """정상화된 데이터를 CSV 파일로 저장"""
    
    if GLOBAL_RESULTS['normalized_df'] is None:
        print("저장할 정상화 데이터가 없습니다. 먼저 run_covid_normalization()을 실행해주세요.")
        return False
    
    if filename is None:
        filename = '외국인입국자_코로나정상화_완료.csv'
    
    try:
        df_normalized = GLOBAL_RESULTS['normalized_df']
        df_normalized.to_csv(filename, index=False, encoding='utf-8')
        print(f"정상화된 데이터가 '{filename}'로 저장되었습니다.")
        print(f"저장된 데이터 크기: {len(df_normalized):,}개 행")
        return True
    except Exception as e:
        print(f"파일 저장 중 오류 발생: {e}")
        return False

def get_model_comparison():
    """모델 비교 결과 반환"""
    
    if GLOBAL_RESULTS['predictions'] is None:
        print("비교할 모델 데이터가 없습니다.")
        return None
    
    valid_predictions = GLOBAL_RESULTS['predictions']
    ensemble_pred = GLOBAL_RESULTS['ensemble_pred']
    
    comparison_data = []
    for name, pred in valid_predictions.items():
        comparison_data.append({
            'Model': name,
            'Mean': np.mean(pred),
            'Std': np.std(pred),
            'Min': np.min(pred),
            'Max': np.max(pred),
            'CV': np.std(pred) / np.mean(pred)
        })
    
    # 앙상블 추가
    comparison_data.append({
        'Model': 'Ensemble_Average',
        'Mean': np.mean(ensemble_pred),
        'Std': np.std(ensemble_pred),
        'Min': np.min(ensemble_pred),
        'Max': np.max(ensemble_pred),
        'CV': np.std(ensemble_pred) / np.mean(ensemble_pred)
    })
    
    return pd.DataFrame(comparison_data)

# 사용 방법 안내
def show_usage():
    """사용 방법 안내"""
    print("="*80)
    print("코로나 정상화 모델 사용 방법")
    print("="*80)
    print("1. run_covid_normalization()           # 모델 학습 및 예측 실행")
    print("2. show_analysis_graphs()              # 분석 그래프 확인")
    print("3. print_detailed_analysis()           # 상세 분석 결과 출력")
    print("4. get_model_comparison()              # 모델 비교 테이블 반환")
    print("5. save_normalized_data()              # 정상화된 CSV 파일 저장")
    print("   save_normalized_data('custom.csv')  # 사용자 지정 파일명")
    print("="*80)
    print("주의: 그래프와 분석 결과를 충분히 검토한 후 파일을 저장하세요!")
    print("="*80)

# 실행 예시
if __name__ == "__main__":
    show_usage()
    print("\n코드가 준비되었습니다.")
    print("run_covid_normalization()를 실행하여 시작하세요!")

GPU 사용 가능: False
Prophet 사용 가능: True
사용 디바이스: cpu
코로나 정상화 모델 사용 방법
1. run_covid_normalization()           # 모델 학습 및 예측 실행
2. show_analysis_graphs()              # 분석 그래프 확인
3. print_detailed_analysis()           # 상세 분석 결과 출력
4. get_model_comparison()              # 모델 비교 테이블 반환
5. save_normalized_data()              # 정상화된 CSV 파일 저장
   save_normalized_data('custom.csv')  # 사용자 지정 파일명
주의: 그래프와 분석 결과를 충분히 검토한 후 파일을 저장하세요!

코드가 준비되었습니다.
run_covid_normalization()를 실행하여 시작하세요!


In [2]:
import sys
import subprocess

print("=" * 60)
print("GPU 및 PyTorch 상태 확인")
print("=" * 60)

# 1. 기본 시스템 정보
print(f"Python 버전: {sys.version}")
print(f"Python 경로: {sys.executable}")

# 2. CUDA 드라이버 확인
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("\n✅ NVIDIA 드라이버 설치됨")
        lines = result.stdout.split('\n')
        for line in lines:
            if 'CUDA Version' in line:
                print(f"CUDA 드라이버 버전: {line.split('CUDA Version: ')[1].split()[0]}")
                break
    else:
        print("❌ NVIDIA 드라이버 문제")
except Exception as e:
    print(f"❌ nvidia-smi 실행 실패: {e}")

# 3. PyTorch 설치 및 CUDA 지원 확인
print("\n" + "-" * 40)
print("PyTorch 상태 확인")
print("-" * 40)

try:
    import torch
    print(f"✅ PyTorch 설치됨: {torch.__version__}")
    
    # CUDA 지원 확인
    print(f"CUDA 지원: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA 버전: {torch.version.cuda}")
        print(f"cuDNN 버전: {torch.backends.cudnn.version()}")
        print(f"GPU 개수: {torch.cuda.device_count()}")
        
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"GPU {i}: {props.name}")
            print(f"  - 메모리: {props.total_memory / 1024**3:.1f} GB")
            print(f"  - 컴퓨트 능력: {props.major}.{props.minor}")
        
        # 현재 GPU 설정
        current_device = torch.cuda.current_device()
        print(f"현재 GPU 디바이스: {current_device}")
        
        # GPU 메모리 사용량
        allocated = torch.cuda.memory_allocated() / 1024**3
        cached = torch.cuda.memory_reserved() / 1024**3
        print(f"할당된 메모리: {allocated:.2f} GB")
        print(f"캐시된 메모리: {cached:.2f} GB")
        
    else:
        print("❌ CUDA 사용 불가능")
        print("가능한 원인:")
        print("1. CUDA 호환 GPU가 없음")
        print("2. NVIDIA 드라이버 설치되지 않음")
        print("3. PyTorch CUDA 버전이 설치되지 않음")
        
except ImportError:
    print("❌ PyTorch가 설치되지 않음")

# 4. GPU 테스트 실행
print("\n" + "-" * 40)
print("GPU 작동 테스트")
print("-" * 40)

try:
    import torch
    if torch.cuda.is_available():
        # 간단한 GPU 연산 테스트
        device = torch.device('cuda')
        
        print("GPU 연산 테스트 시작...")
        x = torch.randn(1000, 1000).to(device)
        y = torch.randn(1000, 1000).to(device)
        
        start_time = torch.cuda.Event(enable_timing=True)
        end_time = torch.cuda.Event(enable_timing=True)
        
        start_time.record()
        z = torch.matmul(x, y)
        end_time.record()
        
        torch.cuda.synchronize()
        elapsed_time = start_time.elapsed_time(end_time)
        
        print(f"✅ GPU 연산 성공!")
        print(f"   1000x1000 행렬 곱셈 시간: {elapsed_time:.2f}ms")
        print(f"   결과 형태: {z.shape}")
        print(f"   GPU 디바이스: {z.device}")
        
        # CPU와 비교
        print("\nCPU vs GPU 성능 비교...")
        x_cpu = torch.randn(1000, 1000)
        y_cpu = torch.randn(1000, 1000)
        
        import time
        start = time.time()
        z_cpu = torch.matmul(x_cpu, y_cpu)
        cpu_time = (time.time() - start) * 1000
        
        print(f"CPU 시간: {cpu_time:.2f}ms")
        print(f"GPU 가속 비율: {cpu_time / elapsed_time:.1f}x")
        
    else:
        print("❌ GPU를 사용할 수 없어 테스트를 건너뜀")
        
except Exception as e:
    print(f"❌ GPU 테스트 실패: {e}")

# 5. 설치된 패키지 확인
print("\n" + "-" * 40)
print("관련 패키지 확인")
print("-" * 40)

packages_to_check = ['torch', 'torchvision', 'torchaudio', 'numpy', 'cuda']
for package in packages_to_check:
    try:
        result = subprocess.run([sys.executable, '-m', 'pip', 'show', package], 
                              capture_output=True, text=True)
        if result.returncode == 0:
            lines = result.stdout.split('\n')
            for line in lines:
                if line.startswith('Version:'):
                    version = line.split(': ')[1]
                    print(f"✅ {package}: {version}")
                    break
        else:
            print(f"❌ {package}: 설치되지 않음")
    except:
        print(f"❌ {package}: 확인 실패")

print("\n" + "=" * 60)
print("GPU 상태 확인 완료")
print("=" * 60)

GPU 및 PyTorch 상태 확인
Python 버전: 3.10.9 | packaged by Anaconda, Inc. | (main, Mar  8 2023, 10:42:25) [MSC v.1916 64 bit (AMD64)]
Python 경로: C:\Users\kmj11\anaconda3\envs\gpudm\python.exe

✅ NVIDIA 드라이버 설치됨
CUDA 드라이버 버전: 12.9

----------------------------------------
PyTorch 상태 확인
----------------------------------------
✅ PyTorch 설치됨: 2.6.0
CUDA 지원: False
❌ CUDA 사용 불가능
가능한 원인:
1. CUDA 호환 GPU가 없음
2. NVIDIA 드라이버 설치되지 않음
3. PyTorch CUDA 버전이 설치되지 않음

----------------------------------------
GPU 작동 테스트
----------------------------------------
❌ GPU를 사용할 수 없어 테스트를 건너뜀

----------------------------------------
관련 패키지 확인
----------------------------------------
✅ torch: 2.6.0
❌ torchvision: 설치되지 않음
❌ torchaudio: 설치되지 않음
✅ numpy: 1.23.5
❌ cuda: 설치되지 않음

GPU 상태 확인 완료
